# 05 — Processing: Alignment, Categorisation, Metrics (interactive)

Per (mode) consolidates inference + LRP + verbalised explanations into one row per sample.
Computes per-sample F1, top-K LRP words via tokenizer offset mapping, GPT word-categorisation
(ENTITY / EMOTIONAL / RHETORIC / CONTENT / OTHER, validated 80.3% on a 61-word sample),
and Jaccard between LRP and LLM word sets.

For grid runs use `run_processing.py` and `run_all.sh`.
Outputs `data/processed/<mode>_processed.{csv,parquet}`.


In [8]:
CONFIG = {
    "dataset_path": "../../datasets/translated/",
    "language": "uk",
    "model_id": "Qwen/Qwen3-VL-4B-Instruct",
    "inference_cache_dir": "./data/inference_cache",
    "lrp_results_dir": "./data/lrp_results",
    "vlm_explanations_dir": "./data/vlm_explanations",
    "output_dir": "./data/processed",
    "modes": ["text", "image", "image+text"],
    "lrp_thresholds": [2, 4],
    "top_k_words": 5,
    "gpt_model": "gpt-4o",
    "max_concurrent": 20,
    "seed": 42,
}

In [9]:
import asyncio
import json
import os
import re
import traceback
from pathlib import Path

import numpy as np
import pandas as pd
from json_repair import repair_json
from openai import AsyncOpenAI
from transformers import AutoProcessor
from tqdm import tqdm

OUT_DIR = Path(CONFIG["output_dir"])
OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR = Path(CONFIG["inference_cache_dir"])
LRP_DIR = Path(CONFIG["lrp_results_dir"])
EXPL_DIR = Path(CONFIG["vlm_explanations_dir"])

api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("OPENAI_API_KEY environment variable must be set.")

processor = AutoProcessor.from_pretrained(CONFIG["model_id"])
client = AsyncOpenAI(api_key=api_key)

In [10]:
STOPWORDS_EN = {"a", "an", "the", "of", "in", "on", "at", "to", "with", "and", "or",
                "is", "are", "was", "be", "this", "that", "it", "its", "by", "for"}
STOPWORDS_UK = {
    'я','ти','ви','він','вона','вони','ми','це','те','ті','мене','мені','мій','моя',
    'не','ні','за','на','що','так','але','та','бо','і','щоб','аби','якщо','коли','чи',
    'то','ще','вже','теж','від','для','до','в','у','з','по','без','через','перед',
    'е','с','нам','наше',
}
STOPWORDS = STOPWORDS_EN | STOPWORDS_UK

def normalize_word(word):
    return re.sub(r"^\W+|\W+$", "", str(word), flags=re.UNICODE).lower().strip()

def safe_set(words, top_k=5):
    seen, out = set(), []
    for w in (words or []):
        n = normalize_word(w)
        if n and n not in STOPWORDS and n not in seen and len(n) > 1:
            seen.add(n); out.append(n)
        if len(out) >= top_k:
            break
    return set(out)

def multilabel_f1(gold_labels, pred_labels):
    gold = set(gold_labels or [])
    pred = set(pred_labels or [])
    denom = len(gold) + len(pred)
    if denom == 0:
        return 1.0
    return 2.0 * len(gold & pred) / denom

In [11]:
def compute_word_relevance_from_offsets(text, relevance_slice):
    """
    Map per-token relevance scores back to words using tokenizer offset mapping.
    Returns dict: word -> mean abs relevance.
    """
    enc = processor.tokenizer(
        text, add_special_tokens=False, return_offsets_mapping=True
    )
    offsets = enc["offset_mapping"]
    n = min(len(offsets), len(relevance_slice))

    word_spans = [(m.start(), m.end(), m.group(0))
                  for m in re.finditer(r"\S+", text)]
    if not word_spans:
        return {}

    text_len = len(text)
    word_vals = {}
    for (tok_start, tok_end), rel in zip(offsets[:n], relevance_slice[:n]):
        if tok_start >= tok_end:
            continue
        mid = max(0, min((tok_start + tok_end - 1) // 2, text_len - 1))
        rel_val = abs(float(rel))
        for w_start, w_end, word in word_spans:
            if w_start <= mid < w_end:
                word_vals.setdefault(word, []).append(rel_val)
                break

    return {w: float(np.mean(v)) for w, v in word_vals.items()}

def get_lrp_top_words(lrp_record, ocr_text, n_words=CONFIG["top_k_words"]):
    """
    Given an LRP record and threshold (fraction of tokens),
    return the set of words whose tokens are in the top-threshold fraction.

    Uses character-level substring search instead of token ID matching to avoid
    BPE context-dependency: the tokenizer merges the preceding space into the
    first token of ocr_text (e.g. ' Дж' != 'Дж'), so exact token ID lookup
    always fails. Decoding the prompt and re-encoding is lossless for Qwen3.
    """
    relevance_norm = lrp_record.get("relevance_norm", [])
    input_len = lrp_record.get("input_len", 0)
    token_ids = lrp_record.get("token_ids", [])

    if not ocr_text.strip() or not relevance_norm:
        return set()

    prompt_ids = token_ids[:input_len]

    # Decode prompt → find ocr_text as a substring (last occurrence)
    prompt_text = processor.tokenizer.decode(prompt_ids, skip_special_tokens=False)
    ocr_start_char = prompt_text.rfind(ocr_text)
    if ocr_start_char < 0:
        return set()
    ocr_end_char = ocr_start_char + len(ocr_text)

    # Re-encode with offset mapping — round-trip is lossless, token count matches
    enc = processor.tokenizer(prompt_text, add_special_tokens=False, return_offsets_mapping=True)
    offsets = enc["offset_mapping"]

    # Collect token indices that overlap the OCR text character span
    ocr_token_indices = [
        i for i, (ts, te) in enumerate(offsets)
        if ts < ocr_end_char and te > ocr_start_char and ts < te and i < len(relevance_norm)
    ]
    if not ocr_token_indices:
        return set()

    abs_rel = [abs(float(relevance_norm[i])) for i in ocr_token_indices]

    word_spans = [(m.start(), m.end(), m.group(0)) for m in re.finditer(r"\S+", ocr_text)]

    word_rel = {}
    for i, idx in enumerate(ocr_token_indices):
        tok_start, tok_end = offsets[idx]
        local_start = max(tok_start, ocr_start_char) - ocr_start_char
        local_end = min(tok_end, ocr_end_char) - ocr_start_char
        local_mid = (local_start + local_end - 1) // 2
        if local_mid < 0 or local_mid >= len(ocr_text):
            continue
        for w_start, w_end, word in word_spans:
            if w_start <= local_mid < w_end:
                word_rel.setdefault(word, []).append(abs_rel[i])
                break

    scored = sorted(
        ((normalize_word(w), float(np.mean(v))) for w, v in word_rel.items()),
        key=lambda x: x[1], reverse=True,
    )
    return set([w for w, _ in scored if w and w not in STOPWORDS and len(w) > 1][:n_words])


In [ ]:
LANG_CAT_NOTE = {
    "uk": "",
    "en": "The meme text is in English. Categorize words in context.",
}

CATEGORIZE_PROMPT = """\
You are analyzing memes. The meme text is in Ukrainian. Categorize words in context.

Meme text: "{meme_text}"
Predicted persuasion techniques: {pred_labels}

The following words were flagged as important in this meme by automated methods:
{word_list}

For each word, assign ONE category based on its role in THIS specific meme:
- ENTITY: named people, places, organizations, brands
- EMOTIONAL: emotionally loaded, fear/hate/outrage-inducing language
- RHETORIC: calls to action, imperatives, slogans, persuasive framing
- CONTENT: neutral factual nouns, events, dates, statistics
- OTHER: numbers alone, punctuation artifacts, unclear

Return ONLY a JSON object mapping each word to its category.
No markdown, no explanation.
Example: {{"word1": "ENTITY", "word2": "EMOTIONAL", "word3": "CONTENT"}}
"""

async def categorize_words(sample_id, meme_text, pred_labels, all_words, semaphore):
    if not all_words:
        return {"id": sample_id, "word_categories": {}}

    lang_note = LANG_CAT_NOTE.get(CONFIG["language"], "")
    prompt = CATEGORIZE_PROMPT.format(
        lang_note=lang_note,
        meme_text=meme_text,
        pred_labels=", ".join(pred_labels),
        word_list=", ".join(sorted(all_words)),
    )

    async with semaphore:
        try:
            resp = await client.chat.completions.create(
                model=CONFIG["gpt_model"],
                messages=[
                    {"role": "system", "content": "You are a linguist. Return only valid JSON."},
                    {"role": "user", "content": prompt},
                ],
                temperature=0,
                max_completion_tokens=300,
            )
            raw = resp.choices[0].message.content.strip().replace("```json", "").replace("```", "").strip()
            word_cats = json.loads(repair_json(raw))
            return {"id": sample_id, "word_categories": word_cats}
        except Exception as e:
            print(f"  Categorize error [{sample_id}]: {e}")
            return {"id": sample_id, "word_categories": {}, "error": str(e)}

In [13]:
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

dataset_path = Path(CONFIG["dataset_path"])
dataset = load_jsonl(dataset_path / "annotations" / "test.jsonl")
id_to_text = {item["id"]: item.get("text", "") for item in dataset}
print(f"Loaded {len(dataset)} samples")

Loaded 158 samples


In [14]:
async def process_mode(mode):
    mode_key = mode.replace("+", "_")
    expl_file = EXPL_DIR / mode_key / "explanations.json"
    lrp_text_dir = LRP_DIR / f"text_{mode_key}"

    explanations = json.load(open(expl_file))
    id_to_expl = {r["id"]: r for r in explanations}

    # Load LRP text results (if available for this mode)
    lrp_text_records = {}
    if lrp_text_dir.exists():
        for p in lrp_text_dir.glob("*.json"):
            if p.name == "failures.json":
                continue
            try:
                r = json.load(open(p))
                lrp_text_records[r["id"]] = r
            except Exception:
                pass

    # ── Word categorization: gather all words per sample ─────────────────────
    semaphore = asyncio.Semaphore(CONFIG["max_concurrent"])
    cat_tasks = []
    sample_words = {}  # id -> {lrp_words, llm_words, all_words}

    for rec in explanations:
        sid = rec["id"]
        meme_text = id_to_text.get(sid, "")
        pred_labels = rec.get("pred_labels", [])
        llm_words = safe_set(rec.get("referred_words", []), top_k=CONFIG["top_k_words"])

        lrp_rec = lrp_text_records.get(sid)
        if lrp_rec and meme_text:
            print(len(llm_words))
            lrp_words = get_lrp_top_words(lrp_rec, meme_text, n_words=len(llm_words))
        else:
            lrp_words = set()

        all_words = llm_words | lrp_words
        sample_words[sid] = {"llm": llm_words, "lrp": lrp_words, "all": all_words}
        cat_tasks.append(categorize_words(sid, meme_text, pred_labels, all_words, semaphore))

    print(f"[{mode}] Categorizing words for {len(cat_tasks)} samples...")
    cat_results = await asyncio.gather(*cat_tasks)
    id_to_cats = {r["id"]: r["word_categories"] for r in cat_results}

    # ── Build rows ────────────────────────────────────────────────────────────
    rows = []
    for rec in tqdm(explanations, desc=f"Build rows [{mode}]"):
        sid = rec["id"]
        meme_text = id_to_text.get(sid, "")
        gold_labels = rec.get("gold_labels", [])
        pred_labels = rec.get("pred_labels", [])

        f1 = multilabel_f1(gold_labels, pred_labels)
        exact = int(set(gold_labels) == set(pred_labels))
        is_trivial = int(len(gold_labels) == 0 and len(pred_labels) == 0)

        llm_words = sample_words.get(sid, {}).get("llm", set())
        word_cats = id_to_cats.get(sid, {})

        lrp_rec = lrp_text_records.get(sid)

        # Jaccard at multiple thresholds
        jaccards = {}
        for thr in CONFIG["lrp_thresholds"]:
            thr_pct = int(thr * 100)
            if lrp_rec and meme_text:
                lrp_w = get_lrp_top_words(lrp_rec, meme_text, n_words=len(llm_words))
            else:
                lrp_w = set()
            inter = llm_words & lrp_w
            union = llm_words | lrp_w
            jaccards[f"jaccard_{thr_pct}"] = len(inter) / len(union) if union else 0.0

        # Word category breakdown
        def count_cat(source_words, category):
            return sum(1 for w in source_words if word_cats.get(w) == category)

        lrp_words = sample_words.get(sid, {}).get("lrp", set())
        row = {
            "sample_id": sid,
            "mode": mode,
            "f1": f1,
            "exact_match": exact,
            "is_trivial": is_trivial,
            "n_gold_labels": len(set(gold_labels)),
            "n_pred_labels": len(set(pred_labels)),
            "techniques": "|".join(sorted(set(gold_labels))),
            "text_length": len(meme_text.split()),
            "has_lrp_text": int(lrp_rec is not None),
            "n_llm_words": len(llm_words),
            "llm_words": list(llm_words),
            "lrp_words": list(lrp_words),
            "word_categories": word_cats,
            # Category counts for LLM words
            "llm_entity": count_cat(llm_words, "ENTITY"),
            "llm_emotional": count_cat(llm_words, "EMOTIONAL"),
            "llm_rhetoric": count_cat(llm_words, "RHETORIC"),
            "llm_content": count_cat(llm_words, "CONTENT"),
            # Category counts for LRP words
            "lrp_entity": count_cat(lrp_words, "ENTITY"),
            "lrp_emotional": count_cat(lrp_words, "EMOTIONAL"),
            "lrp_rhetoric": count_cat(lrp_words, "RHETORIC"),
            "lrp_content": count_cat(lrp_words, "CONTENT"),
            **jaccards,
        }
        rows.append(row)

    df = pd.DataFrame(rows)
    out_csv = OUT_DIR / f"{mode_key}_processed.csv"
    out_parquet = OUT_DIR / f"{mode_key}_processed.parquet"
    df.to_csv(out_csv, index=False)
    df.to_parquet(out_parquet, index=False)
    print(f"[{mode}] Saved {len(df)} rows → {out_parquet}")
    return df

dfs = {}
for mode in CONFIG["modes"]:
    if mode == "image":
        continue
    print(f"\n=== Processing mode: {mode} ===")
    dfs[mode] = await process_mode(mode)
    print(f"[{mode}] Processing complete. DataFrame shape: {dfs[mode].shape}")
print("\nProcessing complete.")


=== Processing mode: text ===
3
3
0
0
3
0
3
5
3
0
5
4
4
3
5
3
4
0
3
3
1
2
0
4
3
0
5
2
0
2
3
2
5
0
5
4
1
0
4
2
2
0
4
3
2
3
0
4
3
0
2
5
4
5
0
0
3
5
5
5
4
5
5
3
0
5
2
4
0
5
0
0
3
5
0
4
5
0
4
2
3
4
3
0
4
2
4
5
4
4
2
5
4
0
5
3
4
5
5
0
0
0
5
4
3
4
3
4
3
4
3
1
4
4
5
4
3
5
4
5
2
4
5
4
2
5
0
3
3
2
4
5
2
3
4
3
5
5
5
4
0
4
5
5
5
5
4
5
5
0
3
3
2
3
0
3
4
5
[text] Categorizing words for 158 samples...


Build rows [text]: 100%|██████████| 158/158 [00:00<00:00, 547.48it/s]


[text] Saved 158 rows → data/processed/text_processed.parquet
[text] Processing complete. DataFrame shape: (158, 24)

=== Processing mode: image+text ===
5
5
2
2
2
3
4
4
4
4
5
4
4
5
5
3
5
3
3
5
3
5
5
5
3
4
5
3
5
2
3
1
5
4
5
4
3
5
5
2
3
3
5
3
5
4
2
5
4
2
3
5
5
5
5
3
4
5
5
5
5
5
5
2
3
5
2
5
3
5
3
5
3
5
2
5
5
3
5
5
5
4
2
3
2
3
5
5
4
3
4
5
5
3
5
3
5
5
5
3
5
4
5
5
5
4
4
5
2
4
4
3
2
5
5
4
5
4
4
5
2
5
4
4
2
5
5
3
5
3
5
5
2
3
4
3
5
5
5
5
5
5
5
4
5
5
4
5
5
3
5
3
3
5
4
4
5
5
[image+text] Categorizing words for 158 samples...


Build rows [image+text]: 100%|██████████| 158/158 [00:00<00:00, 476.61it/s]

[image+text] Saved 158 rows → data/processed/image_text_processed.parquet
[image+text] Processing complete. DataFrame shape: (158, 24)

Processing complete.


In [15]:
for mode, df in dfs.items():
    print(f"\n=== {mode} ===")
    print(df[["f1", "jaccard_25", "jaccard_50", "jaccard_75", "jaccard_100"]].describe().round(3))


=== text ===


KeyError: "['jaccard_25', 'jaccard_50', 'jaccard_75', 'jaccard_100'] not in index"

In [ ]:
from pprint import pprint
for wc in df['word_categories'][:10]:
    pprint(wc)

{}
{'геніальна': 'EMOTIONAL',
 'підтримають': 'RHETORIC',
 'сибіром': 'ENTITY',
 'симоньян': 'ENTITY',
 'таракани': 'EMOTIONAL',
 'українці': 'ENTITY',
 'художник': 'CONTENT',
 'ідея': 'CONTENT'}
{'patriot': 'ENTITY',
 'день': 'CONTENT',
 'зрк': 'ENTITY',
 'кнопку': 'CONTENT',
 'метеорит': 'CONTENT',
 'не тицяли': 'RHETORIC',
 'німеччина': 'ENTITY',
 'передає': 'CONTENT',
 'хтось': 'OTHER'}
{}
{}
{'зараз': 'RHETORIC',
 'повітряну': 'EMOTIONAL',
 'путіну': 'ENTITY',
 'ракети': 'CONTENT',
 'сраку': 'EMOTIONAL',
 'тривогу': 'EMOTIONAL'}
{}
{'боліт': 'CONTENT',
 'будильник': 'CONTENT',
 'офігенні': 'EMOTIONAL',
 'ппо': 'ENTITY',
 'ракети': 'CONTENT',
 'роботу': 'CONTENT'}
{'бензином': 'CONTENT',
 'дивляться': 'CONTENT',
 'домах': 'CONTENT',
 'дрони': 'CONTENT',
 'лупить': 'EMOTIONAL',
 'местное': 'ENTITY',
 'новоросійцовци': 'ENTITY',
 'українські': 'ENTITY',
 'щасливі': 'EMOTIONAL'}
{'дупу': 'EMOTIONAL', 'ракети': 'CONTENT'}


In [ ]:
- ENTITY: named people, places, organizations, brands
- EMOTIONAL: emotionally loaded, fear/hate/outrage-inducing language
- RHETORIC: calls to action, imperatives, slogans, persuasive framing
- CONTENT: neutral factual nouns, events, dates, statistics
- OTHER: numbers alone, punctuation artifacts, unclear

SyntaxError: illegal target for annotation (4014182259.py, line 1)

In [2]:
pred = [{'америка': 'ENTITY',
 'президенти': 'ENTITY',
'проти': 'RHETORIC',
'сказано': 'RHETORIC',
'трампа': 'ENTITY',
'усе': 'RHETORIC',
'цим': 'OTHER'},
{'виженеш': 'RHETORIC',
'вода': 'CONTENT',
'доки': 'OTHER',
'зі': 'OTHER',
'ковбойська': 'EMOTIONAL',
'мудрість': 'EMOTIONAL',
'свиней': 'EMOTIONAL',
'стане': 'OTHER',
'струмка': 'CONTENT',
'чистою': 'EMOTIONAL'},
{'переможець': 'EMOTIONAL', 'фейк': 'EMOTIONAL', 'фейк-переможець': 'EMOTIONAL'},
{'orange': 'CONTENT',
'колір': 'CONTENT',
'літерах': 'CONTENT',
'попросили': 'RHETORIC',
'фрукт': 'CONTENT'},
{'аж': 'OTHER',
'америка': 'ENTITY',
'була': 'CONTENT',
'величною': 'EMOTIONAL',
'гівна': 'EMOTIONAL',
'ніколи': 'RHETORIC',
'повтори': 'RHETORIC',
'такою': 'OTHER',
'шматок': 'EMOTIONAL'},
{'бачиш': 'RHETORIC', 'схему': 'CONTENT', 'тік-так': 'EMOTIONAL'},
{'жінок': 'ENTITY',
'забрати': 'EMOTIONAL',
'них': 'OTHER',
'права': 'CONTENT',
'право': 'CONTENT',
'самозахист': 'CONTENT',
'хочеш': 'RHETORIC'},
{'великий': 'EMOTIONAL',
'вчити': 'RHETORIC',
'йдеш': 'CONTENT',
'куди': 'CONTENT',
'пенсильванію': 'ENTITY',
'птах': 'CONTENT',
'рахувати': 'CONTENT',
'їду': 'CONTENT',
'їх': 'CONTENT'},
{'америка': 'ENTITY',
'жодних': 'RHETORIC',
'трампова': 'ENTITY',
'ілюзій': 'EMOTIONAL'},
{'байдена': 'ENTITY',
'джо': 'ENTITY',
'досягнення': 'CONTENT',
'життєві': 'CONTENT'}]

In [1]:

reclassified = [
    # === BATCH 1 (dicts 1-10) ===
    # 1: "усе що сказано проти Трампа президенти Америка цим..."
    {'америка': 'ENTITY', 'президенти': 'ENTITY', 'проти': 'RHETORIC', 'сказано': 'RHETORIC', 'трампа': 'ENTITY', 'усе': 'OTHER', 'цим': 'OTHER'},
    # 2: cowboy wisdom proverb — "доки виженеш свиней зі струмка вода стане чистою"
    {'виженеш': 'RHETORIC', 'вода': 'CONTENT', 'доки': 'OTHER', 'зі': 'OTHER', 'ковбойська': 'CONTENT', 'мудрість': 'CONTENT', 'свиней': 'EMOTIONAL', 'стане': 'OTHER', 'струмка': 'CONTENT', 'чистою': 'CONTENT'},
    # 3: "фейк-переможець"
    {'переможець': 'CONTENT', 'фейк': 'EMOTIONAL', 'фейк-переможець': 'EMOTIONAL'},
    # 4: orange — color vs fruit
    {'orange': 'CONTENT', 'колір': 'CONTENT', 'літерах': 'CONTENT', 'попросили': 'OTHER', 'фрукт': 'CONTENT'},
    # 5: "Америка ніколи не була такою величною — повтори — шматок гівна"
    {'аж': 'OTHER', 'америка': 'ENTITY', 'була': 'OTHER', 'величною': 'EMOTIONAL', 'гівна': 'EMOTIONAL', 'ніколи': 'RHETORIC', 'повтори': 'RHETORIC', 'такою': 'OTHER', 'шматок': 'EMOTIONAL'},
    # 6: "бачиш схему? тік-так"
    {'бачиш': 'RHETORIC', 'схему': 'CONTENT', 'тік-так': 'EMOTIONAL'},
    # 7: rights / self-defense / women
    {'жінок': 'CONTENT', 'забрати': 'EMOTIONAL', 'них': 'OTHER', 'права': 'CONTENT', 'право': 'CONTENT', 'самозахист': 'CONTENT', 'хочеш': 'RHETORIC'},
    # 8: "їду в Пенсильванію вчити їх рахувати — великий птах"
    {'великий': 'EMOTIONAL', 'вчити': 'RHETORIC', 'йдеш': 'OTHER', 'куди': 'OTHER', 'пенсильванію': 'ENTITY', 'птах': 'CONTENT', 'рахувати': 'CONTENT', 'їду': 'OTHER', 'їх': 'OTHER'},
    # 9: "Трампова Америка — жодних ілюзій"
    {'америка': 'ENTITY', 'жодних': 'RHETORIC', 'трампова': 'ENTITY', 'ілюзій': 'EMOTIONAL'},
    # 10: Joe Biden achievements
    {'байдена': 'ENTITY', 'джо': 'ENTITY', 'досягнення': 'CONTENT', 'життєві': 'CONTENT'},

    # === BATCH 2 (dicts 11-30) ===
    # 11: COVID vaccine trial / lightning / Jimmy
    {'18': 'OTHER', '2020': 'CONTENT', '28': 'OTHER', '4:57': 'OTHER', '72': 'OTHER', '72-річний': 'CONTENT', 'covid': 'CONTENT', 'covid-19': 'CONTENT', 'covid-вакцини': 'CONTENT', 'getty/ap': 'ENTITY', 'moderna': 'ENTITY', 'pm': 'OTHER', 'антивакси': 'EMOTIONAL', 'блискавка': 'CONTENT', 'блискавки': 'CONTENT', 'блискавкою': 'CONTENT', 'був': 'OTHER', 'вакцина': 'CONTENT', 'вакцини': 'CONTENT', 'вас': 'OTHER', 'випробувальної': 'CONTENT', 'випробування': 'CONTENT', 'випробувань': 'CONTENT', 'вражений': 'EMOTIONAL', 'вразливими': 'EMOTIONAL', 'грудня': 'CONTENT', 'джиммі': 'ENTITY', 'днів': 'CONTENT', 'доброволець': 'CONTENT', 'дози': 'CONTENT'},
    # 12: "гей чувакииии вакцина була така погана й можу мандрувати"
    {'була': 'OTHER', 'вакцина': 'CONTENT', 'гей': 'RHETORIC', 'й': 'OTHER', 'мандрувати': 'CONTENT', 'можу': 'OTHER', 'погана': 'EMOTIONAL', 'така': 'OTHER', 'чувакииии': 'RHETORIC'},
    # 13: "більше ніж будь-коли зараз"
    {'будь-коли': 'OTHER', 'більше': 'OTHER', 'зараз': 'OTHER', 'ніж': 'OTHER'},
    # 14: anti-vaxxers / cockpit / YouTube analogy
    {'youtube': 'ENTITY', 'антиваксери': 'EMOTIONAL', 'вильотом': 'CONTENT', 'відео': 'CONTENT', 'завжди': 'OTHER', 'зазирають': 'CONTENT', 'знаю': 'OTHER', 'кажуть': 'OTHER', 'кокпіт': 'CONTENT', 'краще': 'OTHER', 'могли': 'OTHER', 'пасажири': 'CONTENT', 'подивились': 'CONTENT', 'пілотувати': 'CONTENT', 'хто': 'OTHER', 'які': 'OTHER'},
    # 15: masks / child trafficking / COVID conspiracy
    {'66': 'OTHER', '667': 'OTHER', 'covid-19': 'CONTENT', 'буде': 'OTHER', 'ваші': 'OTHER', 'всіх': 'OTHER', 'дитина': 'CONTENT', 'допомагають': 'CONTENT', 'людьми': 'CONTENT', 'маски': 'CONTENT', 'невпізнаними': 'EMOTIONAL', 'непоміченими': 'EMOTIONAL', 'ніж': 'OTHER', 'перевозити': 'CONTENT', 'помре': 'EMOTIONAL', 'продана': 'EMOTIONAL', 'разів': 'CONTENT', 'сша': 'ENTITY', 'того': 'OTHER', 'торгівцям': 'CONTENT', 'частіше': 'OTHER', 'їх': 'OTHER'},
    # 16: "сказали лишуся вдома — педофіли"
    {'вдома': 'CONTENT', 'лишуся': 'RHETORIC', 'педофіли': 'EMOTIONAL', 'сказали': 'OTHER'},
    # 17: "ліберальні ЗМІ оголошують Байдена переможцем — ловлю брехню"
    {'байдена': 'ENTITY', 'брехню': 'EMOTIONAL', 'всіх': 'OTHER', 'змі': 'ENTITY', 'капітане': 'RHETORIC', 'ловлю': 'RHETORIC', 'ліберальні': 'EMOTIONAL', 'оголошують': 'CONTENT', 'переможцем': 'CONTENT', 'частотах': 'CONTENT'},
    # 18: "ніколи не ненавидіти когось — байдуже звідки, як виглядають, кого люблять, кому моляться"
    {'байдуже': 'EMOTIONAL', 'важко': 'EMOTIONAL', 'вам': 'OTHER', 'виглядають': 'CONTENT', 'звідки': 'OTHER', 'казав': 'OTHER', 'кого': 'OTHER', 'когось': 'OTHER', 'кому': 'OTHER', 'люблять': 'EMOTIONAL', 'моляться': 'CONTENT', 'ненавидіти': 'EMOTIONAL', 'ніколи': 'RHETORIC', 'чому': 'OTHER', 'як': 'OTHER'},
    # 19: Dominion ad — "купи 1 — безплатно 10 — лише до 4:00 ранку"
    {'1': 'OTHER', '10': 'OTHER', '4:00': 'OTHER', 'dominion': 'ENTITY', 'безплатно': 'RHETORIC', 'доставка': 'CONTENT', 'купи': 'RHETORIC', 'лише': 'RHETORIC', 'ранку': 'CONTENT'},
    # 20: "стоп кажу — сонний Джо побив рекорд — повна херня"
    {'джо': 'ENTITY', 'кажу': 'RHETORIC', 'побив': 'CONTENT', 'повна': 'EMOTIONAL', 'рекорд': 'CONTENT', 'сонний': 'EMOTIONAL', 'стоп': 'RHETORIC', 'херня': 'EMOTIONAL'},
    # 21: "люблю орлів — такі величні"
    {'величні': 'EMOTIONAL', 'люблю': 'EMOTIONAL', 'орлів': 'ENTITY', 'такі': 'OTHER'},
    # 22: Payday bar name change
    {'payday': 'ENTITY', 'арахісовий': 'CONTENT', 'батончик': 'CONTENT', 'змінює': 'CONTENT', 'карамельний': 'CONTENT', 'назву': 'CONTENT', 'не працює': 'EMOTIONAL', 'ображає': 'EMOTIONAL', 'працює': 'CONTENT', 'тих': 'OTHER', 'хто': 'OTHER'},
    # 23: Epstein island denial
    {'був': 'OTHER', 'епштейна': 'ENTITY', 'жінкою': 'CONTENT', 'мав': 'OTHER', 'ніколи': 'RHETORIC', 'острові': 'ENTITY', 'сексу': 'CONTENT', 'тією': 'OTHER'},
    # 24: PubMed / anti-vaxxer / research
    {'100': 'OTHER', 'pubmed': 'ENTITY', 'антиваксеру': 'EMOTIONAL', 'балів': 'CONTENT', 'вакцини': 'CONTENT', 'досліджував': 'CONTENT', 'жодних': 'RHETORIC', 'знаходжу': 'OTHER', 'казав': 'OTHER', 'мінус': 'CONTENT', 'публікацій': 'CONTENT', 'роками': 'CONTENT', 'твоє': 'OTHER', 'шукаю': 'OTHER', "ім'я": 'CONTENT'},
    # 25: "хрещений шахрай дасть тобі багаторазовий бюлетень — лол"
    {'багаторазовий': 'CONTENT', 'бюлетень': 'CONTENT', 'дасть': 'OTHER', 'лол': 'EMOTIONAL', 'тобі': 'OTHER', 'хрещений': 'ENTITY', 'шахрай': 'EMOTIONAL'},
    # 26: "знаєте чому тепер не морочився з кампанією"
    {'знаєте': 'RHETORIC', 'кампанією': 'CONTENT', 'морочився': 'EMOTIONAL', 'тепер': 'OTHER', 'чому': 'OTHER'},
    # 27: "бля чувак виборці Трампа дізнались як їх кинули"
    {'бля': 'EMOTIONAL', 'виборці': 'CONTENT', 'далеко': 'OTHER', 'дуже': 'OTHER', 'дізнались': 'CONTENT', 'кинули': 'EMOTIONAL', 'ок': 'OTHER', 'почуваються': 'EMOTIONAL', 'після': 'OTHER', 'того': 'OTHER', 'трампа': 'ENTITY', 'чувак': 'EMOTIONAL', 'як': 'OTHER', 'їх': 'OTHER'},
    # 28: "обери 2020 вибори факел"
    {'2020': 'CONTENT', 'вибори': 'CONTENT', 'обери': 'RHETORIC', 'факел': 'EMOTIONAL'},
    # 29: Beetlejuice / Don King / child
    {'бітлджус': 'ENTITY', 'дитину': 'CONTENT', 'дон': 'ENTITY', 'кінг': 'ENTITY', 'мали': 'OTHER'},
    # 30: "приходжу додому з 25 порцій індиком"
    {'25': 'OTHER', '25 порцій': 'CONTENT', 'додому': 'CONTENT', 'порцій': 'CONTENT', 'приходжу': 'CONTENT', 'індиком': 'CONTENT'},
]

In [3]:
total_correct = 0
total = 0
total_words = 0
for sample_x,sample_y in zip(pred,reclassified):
    for word in sample_x:
        if sample_x[word] == sample_y.get(word):
            total_correct += 1
        total += 1
        total_words += 1

accuracy = total_correct / total if total > 0 else 0
print(f"Reclassification accuracy: {accuracy:.2%} ({total_correct}/{total})")
print(f"Total words processed: {total_words}")

Reclassification accuracy: 80.33% (49/61)
Total words processed: 61
